# einops-repeat-broadcast — faded example 3: Broadcast a relative-position bias over heads and batch

> Practice drill from [Delta Drills](https://delta-drills.vercel.app). Atom: `einops-repeat-broadcast`. The last cell reports your progress on the `Einops: Repeat-as-broadcast` subtopic back to Delta Drills.

**Most of the code is already written — complete the one blanked step**, run the test to check it, then run the last cell to record your progress.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Einops: Repeat-as-broadcast` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`einops-repeat-broadcast`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "einops-repeat-broadcast"
DD_SUBTOPIC = "Einops: Repeat-as-broadcast"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

A relative-position bias is one matrix of shape `(T, T)` shared across every head and batch element. `einops.repeat` views it as `(B, H, T, T)` with two new stride-zero leading axes, so it can be added to attention logits without ever materializing `B*H` copies.

## Faded exercise 3

Implement `broadcast_rel_bias(bias, logits)`. `bias` has shape `(T, T)`; `logits` has shape `(B, H, T, T)`. Return `bias` broadcast to `(B, H, T, T)` via `einops.repeat`, inserting the new leading `b` and `h` axes. Complete the blanked `repeat` call.

**Your task:** complete the one blanked step in the code cell below. The surrounding code, function signatures, and variable names are given — work out the missing expression yourself, then run the test.

In [ ]:
import torch as t
import einops
from einops import repeat

t.manual_seed(5)
bias = t.randn(6, 6)
logits = t.randn(2, 3, 6, 6)

def broadcast_rel_bias(bias, logits):
    B, H, T, _ = logits.shape
    bias_b = repeat(bias, 'q k -> b h q k', b=B, h=H)
    return bias_b

print(broadcast_rel_bias(bias, logits).shape)


def _test():
    bias_b = broadcast_rel_bias(bias, logits)
    assert bias_b.shape == (2, 3, 6, 6), bias_b.shape
    for b in range(2):
        for h in range(3):
            assert t.equal(bias_b[b, h], bias), f'slice {b},{h} differs'
    assert bias_b.data_ptr() == bias.data_ptr()
    assert (logits + bias_b).shape == logits.shape


try:
    _test()
    _dd_passed.add('faded3')
    print('[Delta Drills] faded3 passed.')
except AssertionError as _e:
    print('Test failed:', _e)

## Report your progress

Run the cell below to send your progress to Delta Drills. It only counts if the test above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'faded3'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:faded1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()

<details><summary>Solution</summary>

```python
import torch as t
import einops
from einops import repeat

t.manual_seed(5)
bias = t.randn(6, 6)
logits = t.randn(2, 3, 6, 6)

def broadcast_rel_bias(bias, logits):
    B, H, T, _ = logits.shape
    bias_b = repeat(bias, 'q k -> b h q k', b=B, h=H)
    return bias_b

print(broadcast_rel_bias(bias, logits).shape)
```
</details>